In [2]:
from datetime import datetime
import pandas as pd

_tdy = datetime.today().strftime("%Y-%m-%d")

out = pd.read_csv(f"../results/{_tdy}/all_matchups_fg3_predictions.csv")


In [4]:
from __future__ import annotations

import numpy as np
import pandas as pd


# ============================================================
# THREES TICKET SELECTORS
# expects output from predict_game_fg3 (and optionally add_prob_ge_k)
# ============================================================
# Required columns (raw preds):
#   player, team, opp, is_home, pred_fg3a, pred_rate, pred_fg3, baseline_fg3, delta_fg3
# If you ran add_prob_ge_k(out, k=2) / add_prob_ge_k(out, k=3), you also have:
#   p_ge_2, p_ge_3
# ============================================================

def _require_cols(df: pd.DataFrame, req: set[str]) -> None:
    missing = sorted(req - set(df.columns))
    if missing:
        raise ValueError(f"Missing required columns: {missing}")


def add_matchup_key(df: pd.DataFrame) -> pd.DataFrame:
    """Adds matchup_key as AWAY@HOME."""
    out = df.copy()
    out["matchup_key"] = np.where(
        out["is_home"].astype(int) == 1,
        out["opp"].astype(str) + "@" + out["team"].astype(str),   # away@home
        out["team"].astype(str) + "@" + out["opp"].astype(str),   # away@home
    )
    return out


# ------------------------------------------------------------
# 1) "2+" ticket (like your old select_2plus_ticket)
# ------------------------------------------------------------
def select_2plus_ticket(
    df: pd.DataFrame,
    *,
    n_legs: int = 10,
    min_pred_fg3a: float = 5.8,
    min_p_ge_2: float = 0.62,
    min_p_ge_3: float = 0.30,
    max_per_team: int = 3,
    rank_cols: list[str] | None = None,
) -> pd.DataFrame:
    """
    Picks n legs for 2+ threes.
    Filters:
      pred_fg3a >= min_pred_fg3a
      p_ge_2   >= min_p_ge_2
      p_ge_3   >= min_p_ge_3 (optional quality gate)
    Ranks (default): p_ge_2, pred_fg3, pred_fg3a
    """
    req = {"player","team","opp","is_home","pred_fg3a","pred_fg3","p_ge_2","p_ge_3"}
    _require_cols(df, req)

    out = add_matchup_key(df)

    pool = out[
        (out["pred_fg3a"] >= float(min_pred_fg3a)) &
        (out["p_ge_2"] >= float(min_p_ge_2)) &
        (out["p_ge_3"] >= float(min_p_ge_3))
    ].copy()

    if pool.empty:
        return pool

    if rank_cols is None:
        rank_cols = ["p_ge_2", "pred_fg3", "pred_fg3a"]

    pool = pool.sort_values(rank_cols, ascending=[False] * len(rank_cols))

    if max_per_team is not None:
        pool["_team_rank"] = pool.groupby("team").cumcount()
        pool = pool[pool["_team_rank"] < int(max_per_team)].copy()
        pool.drop(columns=["_team_rank"], inplace=True)

    return pool.head(int(n_legs)).reset_index(drop=True)


# ------------------------------------------------------------
# 2) "Jackpot" ticket (+k over baseline)
# ------------------------------------------------------------
def select_jackpot_threes_ticket(
    df: pd.DataFrame,
    *,
    n_legs: int = 3,
    over_baseline_col: str = "p_over_baseline_2",
    min_pred_fg3a: float = 5.5,
    min_p_over_baseline: float = 0.18,
    min_delta_fg3: float = 0.75,
    max_per_team: int = 1,
) -> pd.DataFrame:
    """
    Jackpot = players projected meaningfully above baseline threes.

    Requires columns:
      player, team, opp, is_home, pred_fg3a, pred_fg3, baseline_fg3, delta_fg3, over_baseline_col
    """
    req = {"player","team","opp","is_home","pred_fg3a","pred_fg3","baseline_fg3","delta_fg3", over_baseline_col}
    _require_cols(df, req)

    out = add_matchup_key(df)

    pool = out[
        (out["pred_fg3a"] >= float(min_pred_fg3a)) &
        (out["delta_fg3"] >= float(min_delta_fg3)) &
        (out[over_baseline_col] >= float(min_p_over_baseline))
    ].copy()

    if pool.empty:
        return pool

    pool = pool.sort_values(
        [over_baseline_col, "delta_fg3", "pred_fg3", "pred_fg3a"],
        ascending=[False, False, False, False],
    )

    if max_per_team is not None:
        pool["_team_rank"] = pool.groupby("team").cumcount()
        pool = pool[pool["_team_rank"] < int(max_per_team)].copy()
        pool.drop(columns=["_team_rank"], inplace=True)

    return pool.head(int(n_legs)).reset_index(drop=True)


# ------------------------------------------------------------
# 3) Matchup coverage ticket (2 per matchup + insurance)
# ------------------------------------------------------------
def select_matchup_coverage_threes_ticket(
    df: pd.DataFrame,
    *,
    players_per_matchup: int = 2,
    insurance_per_matchup: int = 1,   # set 0 for "exactly N"
    min_pred_fg3a: float = 5.6,
    min_p_ge_2: float = 0.60,
    min_p_ge_3: float = 0.30,
    max_legs: int | None = None,
) -> pd.DataFrame:
    """
    Matchup-coverage selector (THREES):
      - Groups rows into matchups (AWAY@HOME)
      - Picks top players_per_matchup + insurance_per_matchup per matchup
      - Ranks by: p_ge_2, then pred_fg3, then pred_fg3a

    Requires columns:
      player, team, opp, is_home, pred_fg3a, pred_fg3, p_ge_2, p_ge_3
    """
    req = {"player","team","opp","is_home","pred_fg3a","pred_fg3","p_ge_2","p_ge_3"}
    _require_cols(df, req)

    out = add_matchup_key(df)

    pool = out[
        (out["pred_fg3a"] >= float(min_pred_fg3a)) &
        (out["p_ge_2"] >= float(min_p_ge_2)) &
        (out["p_ge_3"] >= float(min_p_ge_3))
    ].copy()

    if pool.empty:
        return pool

    pool = pool.sort_values(
        ["matchup_key", "p_ge_2", "pred_fg3", "pred_fg3a"],
        ascending=[True, False, False, False],
    )
    pool["_rank"] = pool.groupby("matchup_key").cumcount()

    keep = int(players_per_matchup) + int(insurance_per_matchup)
    ticket = pool[pool["_rank"] < keep].copy()

    if max_legs is not None and len(ticket) > int(max_legs):
        ticket = ticket.sort_values(["p_ge_2","pred_fg3"], ascending=False).head(int(max_legs))

    return (
        ticket
        .drop(columns=["_rank"])
        .sort_values(["matchup_key","p_ge_2","pred_fg3"], ascending=[True, False, False])
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# 4) Pencil labels for threes (3+ / 2+ / coverage_only)
# ------------------------------------------------------------
def assign_pencil_decision_threes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds df['pencil'] labels for threes.

    Requires: p_ge_2, p_ge_3, pred_fg3a
    """
    req = {"pred_fg3a","p_ge_2","p_ge_3"}
    _require_cols(df, req)

    out = df.copy()

    conditions = [
        # Strong 3+ candidates
        (out["p_ge_3"] >= 0.35) & (out["pred_fg3a"] >= 6.0),

        # Solid 2+ candidates
        (out["p_ge_2"] >= 0.60) & (out["pred_fg3a"] >= 4.5),
    ]
    choices = ["3+", "2+"]

    out["pencil"] = np.select(conditions, choices, default="coverage_only")
    return out


# ============================================================
# Example usage
# ============================================================
# out_all = pd.read_csv(f"../results/{_tdy}/all_matchups_fg3_predictions.csv")
# # if you didn't already add these:
# # out_all = add_prob_ge_k(out_all, k=2)
# # out_all = add_prob_ge_k(out_all, k=3)
#
# ticket_2p = select_2plus_ticket(out_all, n_legs=10)
# ticket_cov = select_matchup_coverage_threes_ticket(out_all, players_per_matchup=2, insurance_per_matchup=1)
# labeled = assign_pencil_decision_threes(out_all)
#
# print(ticket_2p[["player","team","opp","is_home","pred_fg3a","pred_rate","pred_fg3","p_ge_2","p_ge_3"]])
# print(ticket_cov[["matchup_key","player","team","opp","is_home","pred_fg3a","pred_rate","pred_fg3","p_ge_2","p_ge_3"]])
# print(labeled[["player","team","pred_fg3a","p_ge_2","p_ge_3","pencil"]].head(25))


In [6]:

out_all = pd.read_csv(f"../results/{_tdy}/all_matchups_fg3_predictions.csv")
# if you didn't already add these:
# out_all = add_prob_ge_k(out_all, k=2)
# out_all = add_prob_ge_k(out_all, k=3)

ticket_2p = select_2plus_ticket(out_all, n_legs=10)
ticket_cov = select_matchup_coverage_threes_ticket(out_all, players_per_matchup=2, insurance_per_matchup=1)
labeled = assign_pencil_decision_threes(out_all)

display(ticket_2p[["player","team","opp","is_home","pred_fg3a","pred_rate","pred_fg3","p_ge_2","p_ge_3"]])
display(ticket_cov[["matchup_key","player","team","opp","is_home","pred_fg3a","pred_rate","pred_fg3","p_ge_2","p_ge_3"]])
display(labeled[["player","team","pred_fg3a","p_ge_2","p_ge_3","pencil"]].head(25))


,player,team,opp,is_home,pred_fg3a,pred_rate,pred_fg3,p_ge_2,p_ge_3
0,Jalen Brunson,NYK,DET,1,7.590769,0.332390,2.523097,0.717410,0.462098
1,Derrick White,BOS,GSW,0,7.672578,0.316933,2.431696,0.698396,0.438550
2,Anfernee Simons,CHI,TOR,1,7.355179,0.325278,2.392475,0.689917,0.428324
3,Tyrese Maxey,PHI,ATL,1,6.457697,0.335094,2.163936,0.636552,0.367601
4,Immanuel Quickley,TOR,CHI,0,6.371519,0.335094,2.135057,0.629318,0.359826
5,Moses Moody,GSW,BOS,1,6.315078,0.337840,2.133489,0.628921,0.359403
6,Jamal Murray,DEN,LAC,0,6.362367,0.335283,2.133191,0.628846,0.359323
7,Russell Westbrook,SAC,ORL,1,6.475645,0.325856,2.110126,0.622982,0.353102
8,Donovan Mitchell,CLE,BKN,1,6.460645,0.325856,2.105238,0.621730,0.351783
9,Duncan Robinson,DET,NYK,0,6.262697,0.335094,2.098592,0.620023,0.349989


,matchup_key,player,team,opp,is_home,pred_fg3a,pred_rate,pred_fg3,p_ge_2,p_ge_3
0,ATL@PHI,Tyrese Maxey,PHI,ATL,1,6.457697,0.335094,2.163936,0.636552,0.367601
1,ATL@PHI,Nickeil Alexander-Walker,ATL,PHI,0,6.585078,0.316933,2.087031,0.617039,0.346866
2,BKN@CLE,Donovan Mitchell,CLE,BKN,1,6.460645,0.325856,2.105238,0.621730,0.351783
3,BOS@GSW,Derrick White,BOS,GSW,0,7.672578,0.316933,2.431696,0.698396,0.438550
4,BOS@GSW,Moses Moody,GSW,BOS,1,6.315078,0.337840,2.133489,0.628921,0.359403
5,DEN@LAC,Jamal Murray,DEN,LAC,0,6.362367,0.335283,2.133191,0.628846,0.359323
6,DET@NYK,Jalen Brunson,NYK,DET,1,7.590769,0.332390,2.523097,0.717410,0.462098
7,DET@NYK,Duncan Robinson,DET,NYK,0,6.262697,0.335094,2.098592,0.620023,0.349989
8,ORL@SAC,Russell Westbrook,SAC,ORL,1,6.475645,0.325856,2.110126,0.622982,0.353102
9,TOR@CHI,Anfernee Simons,CHI,TOR,1,7.355179,0.325278,2.392475,0.689917,0.428324


,player,team,pred_fg3a,p_ge_2,p_ge_3,pencil
0,Tari Eason,HOU,5.175078,0.517281,0.251814,coverage_only
1,Reed Sheppard,HOU,5.258202,0.509970,0.245518,coverage_only
2,Kevin Durant,HOU,5.085769,0.503786,0.240261,coverage_only
3,Jabari Smith Jr.,HOU,4.522578,0.451472,0.198191,coverage_only
4,Alperen Şengün,HOU,1.060645,0.047595,0.005322,coverage_only
5,Amen Thompson,HOU,0.907578,0.034230,0.003202,coverage_only
6,Donovan Mitchell,CLE,6.460645,0.621730,0.351783,3+
7,James Harden,CLE,5.760078,0.574763,0.304430,coverage_only
8,Sam Merrill,CLE,5.415078,0.511775,0.247064,coverage_only
9,Jaylon Tyson,CLE,3.220533,0.290363,0.093827,coverage_only


In [8]:
display(labeled[labeled['pencil'] != 'coverage_only'][["player","team","pred_fg3a","p_ge_2","p_ge_3","pencil"]].head(25))


,player,team,pred_fg3a,p_ge_2,p_ge_3,pencil
6,Donovan Mitchell,CLE,6.460645,0.621730,0.351783,3+
13,Tyrese Maxey,PHI,6.457697,0.636552,0.367601,3+
14,Nickeil Alexander-Walker,ATL,6.585078,0.617039,0.346866,2+
43,Jalen Brunson,NYK,7.590769,0.717410,0.462098,3+
44,Duncan Robinson,DET,6.262697,0.620023,0.349989,2+
56,Anfernee Simons,CHI,7.355179,0.689917,0.428324,3+
57,Immanuel Quickley,TOR,6.371519,0.629318,0.359826,3+
80,Derrick White,BOS,7.672578,0.698396,0.438550,3+
81,Moses Moody,GSW,6.315078,0.628921,0.359403,3+
95,Russell Westbrook,SAC,6.475645,0.622982,0.353102,3+
